# 交互浏览所有保存的 PLY（Plotly）

按你的需求：
- 获取目录下所有 `.ply`
- 逐个展示（3D 可交互）
- `Enter` 进入下一个
- 输入 `esc` 后回车退出


In [1]:
import os
import glob
import numpy as np
import open3d as o3d

try:
    import plotly.graph_objects as go
    import plotly.io as pio
except ImportError as e:
    raise ImportError("未安装 plotly，请先在当前环境执行: pip install plotly") from e

# 在 VSCode / Jupyter 下尽量使用交互 renderer
if "VSCODE_PID" in os.environ:
    pio.renderers.default = "vscode"
else:
    pio.renderers.default = "notebook_connected"

print("plotly renderer =", pio.renderers.default)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
plotly renderer = notebook_connected


In [2]:
PLY_DIR = "/data/haoxiang/logs/task0012-ca-pi-xie/vis_debug_train"
SHOW_LATEST_FIRST = False  # True=最新先看；False=最早先看

ply_files = sorted(glob.glob(os.path.join(PLY_DIR, "*.ply")), reverse=SHOW_LATEST_FIRST)

print(f"PLY 目录: {PLY_DIR}")
print(f"共找到 {len(ply_files)} 个文件")
for i, p in enumerate(ply_files[:20]):
    print(f"[{i}] {os.path.basename(p)}")
if len(ply_files) > 20:
    print(f"... (其余 {len(ply_files)-20} 个省略)")

if len(ply_files) == 0:
    raise FileNotFoundError(f"目录下没有 .ply 文件: {PLY_DIR}")

PLY 目录: /data/haoxiang/logs/task0012-ca-pi-xie/vis_debug_train
共找到 10 个文件
[0] scene_0004_104122060902_1774060267870.ply
[1] scene_0009_104122060902_1774060762823.ply
[2] scene_0017_104122060902_1774061479206.ply
[3] scene_0022_104122060902_1774061859952.ply
[4] scene_0034_104122060902_1774063068653.ply
[5] scene_0036_104122060902_1774063229133.ply
[6] scene_0040_104122060902_1774063705511.ply
[7] scene_0047_104122060902_1774064431477.ply
[8] scene_0050_104122060902_1774064744860.ply
[9] scene_0052_104122060902_1774064886482.ply


In [3]:
def _equal_axis_ranges(points, pad_ratio=0.03):
    xyz_min = points.min(axis=0)
    xyz_max = points.max(axis=0)
    center = (xyz_min + xyz_max) / 2.0
    half = (xyz_max - xyz_min).max() / 2.0
    half = max(float(half), 1e-6) * (1.0 + pad_ratio)
    xr = [center[0] - half, center[0] + half]
    yr = [center[1] - half, center[1] + half]
    zr = [center[2] - half, center[2] + half]
    return xr, yr, zr


def load_ply_points(ply_path, voxel_size=0.0, max_points=220000, seed=0):
    pcd = o3d.io.read_point_cloud(ply_path)

    if voxel_size is not None and voxel_size > 0:
        pcd = pcd.voxel_down_sample(voxel_size=float(voxel_size))

    pts = np.asarray(pcd.points)
    cols = np.asarray(pcd.colors)

    if pts.size == 0:
        raise ValueError(f"点云为空: {ply_path}")

    has_rgb = (cols.ndim == 2 and cols.shape[0] == pts.shape[0] and cols.shape[1] == 3)
    if not has_rgb:
        cols = None

    if max_points is not None and pts.shape[0] > int(max_points):
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=int(max_points), replace=False)
        pts = pts[idx]
        if cols is not None:
            cols = cols[idx]

    return pts, cols


def plot_ply_interactive(
    ply_path,
    voxel_size=0.003,
    max_points=220000,
    point_size=1.3,
    opacity=0.9,
    color_mode="z",   # "z" 或 "rgb"
    z_clip=None,
    show_axes=True,
):
    pts, cols = load_ply_points(
        ply_path,
        voxel_size=voxel_size,
        max_points=max_points,
        seed=0,
    )

    if z_clip is not None:
        zmin, zmax = float(z_clip[0]), float(z_clip[1])
        keep = (pts[:, 2] >= zmin) & (pts[:, 2] <= zmax)
        pts = pts[keep]
        if cols is not None:
            cols = cols[keep]

    if pts.shape[0] == 0:
        raise ValueError("过滤后无点，请放宽 z_clip 或调大 max_points")

    if color_mode == "rgb" and cols is not None:
        rgb = np.clip((cols * 255.0).astype(np.uint8), 0, 255)
        marker_color = [f"rgb({r},{g},{b})" for r, g, b in rgb]
        marker = dict(size=point_size, opacity=opacity, color=marker_color)
    else:
        marker = dict(
            size=point_size,
            opacity=opacity,
            color=pts[:, 2],
            colorscale="Turbo",
            colorbar=dict(title="z"),
        )

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            mode="markers",
            marker=marker,
            name="pointcloud",
        )
    )

    xr, yr, zr = _equal_axis_ranges(pts)

    if show_axes:
        c = pts.mean(axis=0)
        L = (xr[1] - xr[0]) * 0.12
        fig.add_trace(go.Scatter3d(x=[c[0], c[0]+L], y=[c[1], c[1]],   z=[c[2], c[2]],   mode="lines", line=dict(color="red", width=6),   name="+X"))
        fig.add_trace(go.Scatter3d(x=[c[0], c[0]],   y=[c[1], c[1]+L], z=[c[2], c[2]],   mode="lines", line=dict(color="green", width=6), name="+Y"))
        fig.add_trace(go.Scatter3d(x=[c[0], c[0]],   y=[c[1], c[1]],   z=[c[2], c[2]+L], mode="lines", line=dict(color="blue", width=6),  name="+Z"))

    fig.update_layout(
        title=f"{os.path.basename(ply_path)} | shown={pts.shape[0]:,}",
        template="plotly_white",
        margin=dict(l=0, r=0, b=0, t=45),
        scene=dict(
            xaxis=dict(title="x", range=xr, showbackground=True, backgroundcolor="rgb(245,245,245)"),
            yaxis=dict(title="y", range=yr, showbackground=True, backgroundcolor="rgb(245,245,245)"),
            zaxis=dict(title="z", range=zr, showbackground=True, backgroundcolor="rgb(245,245,245)"),
            aspectmode="cube",
        ),
        legend=dict(x=0.01, y=0.99),
    )

    fig.show()
    return fig

In [4]:
# 逐个浏览：回车看下一个；输入 esc 后回车立即退出（不再显示下一个）
# 每次进入下一轮前会清空上一轮输出。
# 注意：Jupyter 的 input 无法捕获“单独按 Esc 键”，需要输入 esc 再回车。
from IPython.display import clear_output

stop_browsing = False
for i, ply_path in enumerate(ply_files):
    clear_output(wait=True)
    print("\n" + "=" * 100)
    print(f"[{i+1}/{len(ply_files)}] {ply_path}")

    plot_ply_interactive(
        ply_path,
        voxel_size=0.0025,
        max_points=220000,
        point_size=1.2,
        opacity=0.9,
        color_mode="rgb",
        z_clip=None,
        show_axes=True,
    )

    if i == len(ply_files) - 1:
        print("已到最后一个文件。")
        break

    while True:
        cmd = input("按 Enter 查看下一个；输入 esc 退出：").strip().lower()
        if cmd == "":
            break
        if cmd == "esc":
            stop_browsing = True
            break
        print("无效输入：直接回车看下一个，或输入 esc 退出。")

    if stop_browsing:
        print("用户终止浏览。")
        break


[4/10] /data/haoxiang/logs/task0012-ca-pi-xie/vis_debug_train/scene_0022_104122060902_1774061859952.ply


KeyboardInterrupt: 